# Sequence Motifs and Protein Domains

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain what sequence motifs are and why they matter biologically
- Construct and use Position Weight Matrices (PWMs) to score sequences
- Understand information content and sequence logos
- Work with PROSITE patterns
- Describe protein domains and the major domain databases (Pfam, InterPro, CDD)
- Search for domains using hmmscan and InterProScan
- Interpret domain architectures of multi-domain proteins
- Apply these concepts to find transcription factor binding sites and annotate protein domains

## Why this notebook matters

Protein domains are the modular units of protein architecture and function. Most eukaryotic proteins are multi-domain, and the same domain can appear in thousands of proteins across all forms of life. Understanding domain databases (Pfam, InterPro, SMART), profile hidden Markov models (HMMs), and domain architecture analysis is essential for annotating novel proteins, predicting function, and understanding protein evolution through domain shuffling and fusion events.

## How to use this notebook

1. Run cells top-to-bottom in order — later cells depend on earlier ones
2. Run the environment check cell first to confirm all imports work
3. Read the explanatory text before each code cell — the context matters
4. The exercises at the end are designed to be done after reading each section

## Complicated moments explained

- **Domain vs. motif**: A domain is a structurally and functionally independent unit, typically 50-300 residues, that can fold independently and often recurs in diverse protein contexts. A motif is a shorter pattern (typically <20 residues) that is functionally important but does not necessarily fold independently.
- **Profile HMM scoring**: Pfam represents each domain family as a profile HMM with match states, insert states, and delete states. Scores are in bits relative to a null model; the gathering threshold (GA) is family-specific — do not apply a universal bit-score cutoff.
- **E-value vs. score for domain search**: hmmscan reports two E-values: sequence E-value (is there any domain match in the whole sequence?) and domain E-value (how significant is this specific domain hit?). For annotating individual domains, use the domain E-value.
- **Domain boundaries are approximate**: Pfam domain boundaries represent the HMM consensus. Always check the actual alignment to determine the true structural domain extents for your protein.
- **InterPro integrates multiple databases**: An InterPro entry (IPR000xxx) integrates matching entries from Pfam, SMART, ProSite, PRINTS, CDD, and SUPERFAMILY. InterProScan is the best first-pass tool for comprehensive domain annotation.

## Environment check (run this first)

In [ ]:
# Environment check
import numpy as np
import matplotlib.pyplot as plt
import shutil

print("Imports ready.")

hmmer_found = shutil.which('hmmscan')
if hmmer_found:
    print(f"HMMER hmmscan found at: {hmmer_found}")
else:
    print("HMMER not found (install: conda install -c bioconda hmmer)")
    print("The notebook uses simulated HMMER output for demonstrations.")

print("\nDomain databases:")
databases = [
    ("Pfam",     "Profile HMMs, ~20,000 families",      "Best overall coverage"),
    ("InterPro", "Integrates Pfam, SMART, CDD, etc.",   "Most comprehensive"),
    ("SMART",    "Focus on signaling domains",           "Excellent for kinases, receptors"),
    ("CDD",      "NCBI-curated, integrates Pfam/SMART", "Free with local BLAST"),
    ("PROSITE",  "Patterns and profiles",                "Motifs and active sites"),
]
for db, content, notes in databases:
    print(f"  {db:<12} {content:<42} {notes}")

In [ ]:
def display_ppm_heatmap(ppm, title="Position Probability Matrix"):
    """Display the PPM as a heatmap."""
    fig, ax = plt.subplots(figsize=(max(6, ppm.shape[1] * 0.8), 2.5))
    im = ax.imshow(ppm, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)

    ax.set_yticks(range(4))
    ax.set_yticklabels(BASES)
    ax.set_xticks(range(ppm.shape[1]))
    ax.set_xticklabels([str(i + 1) for i in range(ppm.shape[1])])
    ax.set_xlabel("Position")
    ax.set_title(title)

    # Annotate cells
    for i in range(4):
        for j in range(ppm.shape[1]):
            color = "white" if ppm[i, j] > 0.6 else "black"
            ax.text(j, i, f"{ppm[i, j]:.2f}", ha="center", va="center",
                    fontsize=8, color=color)

    plt.colorbar(im, ax=ax, label="Probability")
    plt.tight_layout()
    plt.show()


display_ppm_heatmap(ppm, "PPM for TF Binding Sites")

### 2.4 Scoring Sequences with a PWM

In [ ]:
def score_sequence(pwm, sequence):
    """Score a sequence against a PWM. Returns sum of log-odds scores."""
    score = 0.0
    for pos, base in enumerate(sequence.upper()):
        if base in BASES:
            score += pwm[BASES.index(base), pos]
    return score


def scan_sequence(pwm, sequence, threshold=None):
    """
    Scan a long sequence for PWM matches.
    Returns list of (position, subsequence, score) sorted by score descending.
    """
    motif_len = pwm.shape[1]
    max_score = np.sum(np.max(pwm, axis=0))
    if threshold is None:
        threshold = 0.6 * max_score  # default: 60% of max

    hits = []
    for i in range(len(sequence) - motif_len + 1):
        subseq = sequence[i:i + motif_len]
        s = score_sequence(pwm, subseq)
        if s >= threshold:
            hits.append((i, subseq, s))

    return sorted(hits, key=lambda x: -x[2])


# Test scoring
test_seqs = ["ATGACTCA", "GTGACTCA", "CCCCCCCC", "ATGACTTA"]
max_s = np.sum(np.max(pwm, axis=0))

print(f"{'Sequence':<12} {'Score':>7} {'% of max':>9}")
print("-" * 30)
for seq in test_seqs:
    s = score_sequence(pwm, seq)
    print(f"{seq:<12} {s:7.2f} {100 * s / max_s:8.1f}%")

In [ ]:
# Scan a longer sequence
target = "GCCTAGATGACTCAGGTTTCCCGTGACTCAATGCAATGACTTACCC"

hits = scan_sequence(pwm, target, threshold=0.5 * np.sum(np.max(pwm, axis=0)))

print(f"Scanning: {target}")
print(f"\n{'Pos':>4} {'Subseq':<10} {'Score':>7}")
print("-" * 25)
for pos, subseq, score in hits:
    print(f"{pos:4d} {subseq:<10} {score:7.2f}")

### 2.5 Both-Strand Scanning

Transcription factor binding sites can occur on either DNA strand. To search both
strands, also scan the reverse complement.

In [ ]:
def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence."""
    comp = str.maketrans("ATGCatgc", "TACGtacg")
    return seq.translate(comp)[::-1]


def scan_both_strands(pwm, sequence, threshold=None):
    """Scan both strands for PWM matches."""
    fwd_hits = scan_sequence(pwm, sequence, threshold)
    rc_seq = reverse_complement(sequence)
    rev_hits = scan_sequence(pwm, rc_seq, threshold)

    # Convert reverse hit positions to forward-strand coordinates
    motif_len = pwm.shape[1]
    seq_len = len(sequence)
    all_hits = [(pos, subseq, score, "+") for pos, subseq, score in fwd_hits]
    for pos, subseq, score in rev_hits:
        fwd_pos = seq_len - pos - motif_len
        all_hits.append((fwd_pos, subseq, score, "-"))

    return sorted(all_hits, key=lambda x: -x[2])


dual_hits = scan_both_strands(pwm, target,
                               threshold=0.5 * np.sum(np.max(pwm, axis=0)))

print(f"{'Pos':>4} {'Strand':>6} {'Match':<10} {'Score':>7}")
print("-" * 32)
for pos, subseq, score, strand in dual_hits:
    print(f"{pos:4d} {strand:>6} {subseq:<10} {score:7.2f}")

---

## 3. Information Content and Sequence Logos

### 3.1 Information Content

The **information content** (IC) at each position quantifies how conserved it is:

$$IC_i = \log_2(K) - H_i = \log_2(K) + \sum_{b} p_{b,i} \log_2(p_{b,i})$$

For DNA ($K=4$): $IC$ ranges from 0 bits (uniform, no conservation) to 2 bits
(perfectly conserved).

The **total information content** of a motif is $\sum_i IC_i$ and reflects how specific
the motif is overall.

In [ ]:
def information_content(ppm):
    """Calculate information content per position (DNA: max 2 bits)."""
    ic = np.zeros(ppm.shape[1])
    for pos in range(ppm.shape[1]):
        probs = ppm[:, pos]
        entropy = -np.sum(probs * np.log2(probs + 1e-12))
        ic[pos] = 2.0 - entropy  # 2 = log2(4)
    return ic


ic = information_content(ppm)

print(f"{'Position':>8} {'IC (bits)':>10} {'Bar'}")
print("-" * 35)
for i, val in enumerate(ic):
    bar = '#' * int(val * 15)
    print(f"{i + 1:8d} {val:10.3f} {bar}")
print(f"\nTotal IC: {ic.sum():.2f} bits (max possible: {2 * len(ic):.0f} bits)")

### 3.2 Sequence Logos

A **sequence logo** is the standard visualization of a motif. At each position:
- The total height of the stack equals the information content (in bits)
- Each letter's height is proportional to its frequency: $h_{b,i} = p_{b,i} \times IC_i$
- Letters are stacked from most to least frequent (top to bottom)

We can create a basic logo with matplotlib (for publication quality, use
[WebLogo](https://weblogo.berkeley.edu/) or the `logomaker` Python package).

In [ ]:
BASE_COLORS = {"A": "green", "C": "blue", "G": "orange", "T": "red"}


def plot_sequence_logo(ppm, title="Sequence Logo"):
    """
    Plot a simple sequence logo using stacked colored bars.

    Each bar's height = p(base, pos) * IC(pos).
    """
    ic = information_content(ppm)
    n_pos = ppm.shape[1]

    fig, ax = plt.subplots(figsize=(max(5, n_pos * 0.7), 3))

    for pos in range(n_pos):
        # Sort bases by probability at this position
        order = np.argsort(ppm[:, pos])  # ascending
        y_bottom = 0
        for idx in order:
            height = ppm[idx, pos] * ic[pos]
            if height > 0.01:
                base = BASES[idx]
                ax.bar(pos + 1, height, bottom=y_bottom, width=0.8,
                       color=BASE_COLORS[base], edgecolor="none")
                if height > 0.15:
                    ax.text(pos + 1, y_bottom + height / 2, base,
                            ha="center", va="center", fontweight="bold",
                            fontsize=10, color="white")
                y_bottom += height

    ax.set_xlim(0.4, n_pos + 0.6)
    ax.set_ylim(0, 2.1)
    ax.set_xlabel("Position")
    ax.set_ylabel("Information content (bits)")
    ax.set_title(title)
    ax.set_xticks(range(1, n_pos + 1))
    plt.tight_layout()
    plt.show()


plot_sequence_logo(ppm, "Sequence Logo: Example TF Binding Motif")

---

## 4. PROSITE Patterns

[PROSITE](https://prosite.expasy.org/) is a database of protein families and domains
described as **regular-expression-like patterns** or **profiles**.

### PROSITE Pattern Syntax

| Syntax | Meaning | Example |
|--------|---------|---------|
| Uppercase letter | Specific amino acid | `C` means cysteine |
| `x` | Any amino acid | `x` matches anything |
| `[ABC]` | One of the listed | `[LIVMF]` = hydrophobic |
| `{P}` | Not proline | `{PC}` = not Pro or Cys |
| `(3)` | Repeat 3 times | `x(3)` = any 3 residues |
| `(2,4)` | Repeat 2-4 times | `x(2,4)` = 2 to 4 residues |
| `-` | Separator | between elements |

**Example:** N-glycosylation site: `N-{P}-[ST]-{P}`  
Meaning: Asn, then any residue except Pro, then Ser or Thr, then any except Pro.

### Converting PROSITE to Python Regex

In [ ]:
def prosite_to_regex(pattern):
    """
    Convert a PROSITE pattern to a Python regular expression.

    Handles: specific residues, x (any), [set], {exclusion}, (n), (n,m), separators.
    """
    # Remove terminal dots if present
    pattern = pattern.strip(".")
    elements = pattern.split("-")
    regex_parts = []

    for elem in elements:
        # Check for repeat quantifier
        match_rep = re.match(r'^(.+?)\((\d+)(?:,(\d+))?\)$', elem)

        if match_rep:
            core = match_rep.group(1)
            low = match_rep.group(2)
            high = match_rep.group(3)
        else:
            core = elem
            low = high = None

        # Convert core
        if core == "x":
            r = "."
        elif core.startswith("[") and core.endswith("]"):
            r = core  # already valid regex character class
        elif core.startswith("{") and core.endswith("}"):
            # Exclusion set -> negative character class
            excluded = core[1:-1]
            r = f"[^{excluded}]"
        elif len(core) == 1 and core.isalpha():
            r = core
        else:
            r = core  # fallback

        # Add quantifier
        if low is not None:
            if high is not None:
                r += f"{{{low},{high}}}"
            else:
                r += f"{{{low}}}"

        regex_parts.append(r)

    return "".join(regex_parts)


# Examples
patterns = {
    "N-glycosylation": "N-{P}-[ST]-{P}",
    "Protein kinase C phosphorylation": "[ST]-x-[RK]",
    "Zinc finger C2H2": "C-x(2,4)-C-x(3)-[LIVMFYWC]-x(8)-H-x(3,5)-H",
}

for name, pat in patterns.items():
    regex = prosite_to_regex(pat)
    print(f"{name}:")
    print(f"  PROSITE: {pat}")
    print(f"  Regex:   {regex}\n")

In [ ]:
def scan_prosite(sequence, prosite_pattern, pattern_name="pattern"):
    """Scan a protein sequence for a PROSITE pattern."""
    regex = prosite_to_regex(prosite_pattern)
    matches = []

    for m in re.finditer(regex, sequence):
        matches.append((m.start() + 1, m.end(), m.group()))  # 1-based

    print(f"Pattern '{pattern_name}' ({prosite_pattern}):")
    if matches:
        for start, end, matched in matches:
            print(f"  Position {start}-{end}: {matched}")
    else:
        print("  No matches found.")

    return matches


# Scan for N-glycosylation sites in a protein
test_protein = "MKVLLFAANISTHRGALVNVTPKSC" \
               "NLTKVDYKNQTLLGSNLSECVFAIDNATASEKFLNYTRAR"

scan_prosite(test_protein, "N-{P}-[ST]-{P}", "N-glycosylation")
print()
scan_prosite(test_protein, "[ST]-x-[RK]", "PKC phosphorylation")

---

## 5. Protein Domains

### 5.1 What Are Protein Domains?

A **protein domain** is a conserved, independently-folding structural/functional unit
within a protein. Most proteins in complex organisms are **multi-domain** proteins.

```
Example: Src kinase (536 aa)

N-term                                                             C-term
  |----[  SH3  ]---[    SH2    ]---[      Kinase domain      ]---|
       84-145       151-248         270-520

Key properties:
  - Each domain folds independently
  - Each has a distinct function (SH3: binds proline-rich peptides,
    SH2: binds phosphotyrosine, Kinase: catalyzes phosphorylation)
  - Domains are found in many different proteins (domain shuffling)
  - Domains are conserved across species
```

### 5.2 Domain Databases

| Database | Focus | Method |
|----------|-------|--------|
| **Pfam** | Protein families and domains | Hidden Markov Models (HMMs) |
| **InterPro** | Integrated annotations from 13+ databases | Consortium |
| **CDD** (NCBI) | Conserved Domain Database | PSSMs + HMMs |
| **SMART** | Signaling / extracellular domains | HMMs |
| **PROSITE** | Patterns and profiles | Regex + profiles |
| **SUPERFAMILY** | Structural superfamilies (SCOP) | HMMs |
| **Gene3D** | Structural domains (CATH) | HMMs |

**InterPro** is the most comprehensive: it integrates results from all the above into
unified **IPR accessions** linked to Gene Ontology (GO) terms.

In [ ]:
# Common domain reference data
DOMAIN_DB = {
    "PF00069": {"name": "Protein kinase (Pkinase)", "interpro": "IPR000719",
                "function": "Phosphorylation of substrates",
                "avg_length": 260},
    "PF00017": {"name": "SH2 domain", "interpro": "IPR000980",
                "function": "Binds phosphotyrosine peptides",
                "avg_length": 100},
    "PF00018": {"name": "SH3 domain", "interpro": "IPR001452",
                "function": "Binds proline-rich sequences",
                "avg_length": 60},
    "PF00096": {"name": "Zinc finger C2H2", "interpro": "IPR007087",
                "function": "DNA binding (transcription factors)",
                "avg_length": 23},
    "PF00400": {"name": "WD40 repeat", "interpro": "IPR001680",
                "function": "Protein-protein interaction scaffold",
                "avg_length": 40},
    "PF00047": {"name": "Immunoglobulin domain", "interpro": "IPR013151",
                "function": "Structural/recognition domain",
                "avg_length": 80},
    "PF07714": {"name": "Pkinase_Tyr (Tyrosine kinase)", "interpro": "IPR001245",
                "function": "Tyrosine-specific phosphorylation",
                "avg_length": 260},
}

print(f"{'Pfam':>8}  {'Name':<30}  {'IPR':<12}  {'Avg len'}")
print("-" * 70)
for acc, info in DOMAIN_DB.items():
    print(f"{acc:>8}  {info['name']:<30}  {info['interpro']:<12}  ~{info['avg_length']} aa")

### 5.3 Profile Hidden Markov Models (profile HMMs)

Pfam uses **profile HMMs** to define each domain family. A profile HMM is built from
a multiple sequence alignment of known family members.

```
Profile HMM architecture:

Begin --> M1 --> M2 --> M3 --> ... --> Mn --> End
           |      |      |              |
          I1     I2     I3             In   (Insert states)
           |      |      |              |
          D1     D2     D3             Dn   (Delete states)

M (Match):  emit amino acid with position-specific probabilities
I (Insert): emit any amino acid (handle insertions in query)
D (Delete): silent state (handle deletions / gaps in query)
```

The tool **HMMER** (hmmer.org) implements profile HMM search:
- `hmmscan`: search a **sequence** against a **database of HMMs** (e.g., Pfam)
- `hmmsearch`: search an **HMM** against a **sequence database**

**E-value interpretation:**
- E < 0.01: significant hit (likely a true domain)
- E > 1: not significant

---

## 6. Searching for Domains: hmmscan and InterProScan

### 6.1 Using hmmscan (Command Line)

```bash
# Download Pfam HMM library
wget https://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.hmm.gz
gunzip Pfam-A.hmm.gz
hmmpress Pfam-A.hmm

# Scan a protein sequence
hmmscan --domtblout results.domtab Pfam-A.hmm query.fasta
```

### 6.2 Using InterProScan

InterProScan combines Pfam, PROSITE, SMART, and other databases in one search:

```bash
# Local installation
interproscan.sh -i query.fasta -f tsv,gff3 -goterms -pa
```

Or use the [web interface](https://www.ebi.ac.uk/interpro/search/sequence/).

### 6.3 Parsing Domain Scan Results

In [ ]:
def parse_domtblout(text):
    """
    Parse HMMER domtblout format.

    Returns list of dicts with domain hit information.
    """
    hits = []
    for line in text.strip().split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        fields = line.split()
        if len(fields) >= 23:
            hits.append({
                "domain_name": fields[0],
                "domain_acc": fields[1],
                "query_name": fields[3],
                "e_value": float(fields[6]),
                "score": float(fields[7]),
                "dom_e_value": float(fields[12]),
                "dom_score": float(fields[13]),
                "ali_from": int(fields[17]),
                "ali_to": int(fields[18]),
                "env_from": int(fields[19]),
                "env_to": int(fields[20]),
                "description": " ".join(fields[22:]),
            })
    return sorted(hits, key=lambda h: h["ali_from"])


# Simulated hmmscan output for human Src kinase
example_domtblout = """# --- hmmscan domtblout ---
SH3_1         PF00018.20  1  Src_Human  PF00018.20   1.2e-15  55.3  0.1  1.8e-15  54.8  0.1  1  1  84  145  83  146  0.98  SH3 domain
SH2           PF00017.22  1  Src_Human  PF00017.22   3.4e-30  102.1  0.2  5.1e-30  101.5  0.1  1  1  151  248  150  249  0.99  SH2 domain
Pkinase_Tyr   PF07714.18  1  Src_Human  PF07714.18   2.1e-75  252.8  0.3  3.2e-75  252.2  0.2  1  1  270  520  268  521  0.99  Protein tyrosine kinase
"""

domains = parse_domtblout(example_domtblout)

print(f"{'Domain':<15} {'Pfam':<13} {'Start':>5} {'End':>5} {'E-value':>10} {'Score':>7}")
print("-" * 60)
for d in domains:
    print(f"{d['domain_name']:<15} {d['domain_acc']:<13} "
          f"{d['ali_from']:5d} {d['ali_to']:5d} "
          f"{d['e_value']:10.1e} {d['score']:7.1f}")

---

## 7. Domain Architectures and Multi-Domain Proteins

The arrangement of domains in a protein is called its **domain architecture**.
Proteins with the same domain architecture often share function, even across species.

Domain architectures evolve through:
- **Domain duplication** (e.g., tandem repeats of Ig domains in titin)
- **Domain shuffling** (recombination puts domains in new contexts)
- **Domain loss** or **gain** during evolution

### Examples of Domain Architectures

```
Src kinase:      [SH3]--[SH2]--[Kinase]
p53:             [TAD]--[PRD]--[DBD]--[OD]
Titin:           [Ig]--[Ig]--[FnIII]--[Ig]--[Ig]--[FnIII]--...(>300 domains)
Antibody (IgG):  [VH]--[CH1]--[CH2]--[CH3]   (heavy chain)
```

In [ ]:
def visualize_domain_architecture(protein_name, protein_length, domains,
                                   figsize=(12, 1.5)):
    """
    Draw domain architecture as a horizontal bar chart.

    domains: list of dicts with keys 'name', 'start', 'end', and optional 'color'.
    """
    colors_palette = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2",
                      "#59a14f", "#edc948", "#b07aa1", "#ff9da7"]

    fig, ax = plt.subplots(figsize=figsize)

    # Draw backbone
    ax.plot([1, protein_length], [0, 0], color="gray", linewidth=3, solid_capstyle="round")

    for i, dom in enumerate(domains):
        color = dom.get("color", colors_palette[i % len(colors_palette)])
        start = dom["start"]
        width = dom["end"] - dom["start"]
        ax.barh(0, width, left=start, height=0.5, color=color,
                edgecolor="black", linewidth=0.8)
        mid = start + width / 2
        ax.text(mid, 0, dom["name"], ha="center", va="center",
                fontsize=9, fontweight="bold", color="white")
        ax.text(mid, -0.4, f"{start}-{dom['end']}", ha="center",
                va="top", fontsize=7, color="gray")

    ax.set_xlim(0, protein_length + 10)
    ax.set_ylim(-0.7, 0.7)
    ax.set_xlabel("Residue position")
    ax.set_title(f"{protein_name} ({protein_length} aa)")
    ax.set_yticks([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    plt.tight_layout()
    plt.show()


# Src kinase
visualize_domain_architecture("Human Src kinase", 536, [
    {"name": "SH3", "start": 84, "end": 145},
    {"name": "SH2", "start": 151, "end": 248},
    {"name": "Pkinase_Tyr", "start": 270, "end": 520},
])

In [ ]:
# More examples
visualize_domain_architecture("p53 tumor suppressor", 393, [
    {"name": "TAD", "start": 1, "end": 61},
    {"name": "PRD", "start": 64, "end": 92},
    {"name": "DBD", "start": 102, "end": 292},
    {"name": "OD", "start": 323, "end": 356},
])

visualize_domain_architecture("BRCA1", 1863, [
    {"name": "RING", "start": 1, "end": 109},
    {"name": "BRCT", "start": 1642, "end": 1736},
    {"name": "BRCT", "start": 1756, "end": 1855},
])

In [ ]:
def compare_architectures(proteins):
    """
    Compare domain architectures of multiple proteins.

    proteins: list of dicts with keys 'name', 'length', 'domains'
    """
    # Collect all unique domain names
    all_domains = set()
    for prot in proteins:
        for dom in prot["domains"]:
            all_domains.add(dom["name"])

    print(f"{'Protein':<20} {'Length':>6}  Architecture")
    print("-" * 70)
    for prot in proteins:
        doms = sorted(prot["domains"], key=lambda d: d["start"])
        arch_str = "--".join(f"[{d['name']}]" for d in doms)
        print(f"{prot['name']:<20} {prot['length']:>6}  {arch_str}")

    # Shared domains
    print(f"\nShared domains across all proteins:")
    shared = all_domains.copy()
    for prot in proteins:
        prot_doms = {d["name"] for d in prot["domains"]}
        shared &= prot_doms

    if shared:
        print(f"  {', '.join(sorted(shared))}")
    else:
        print("  None")


kinase_family = [
    {"name": "Src", "length": 536,
     "domains": [{"name": "SH3", "start": 84, "end": 145},
                 {"name": "SH2", "start": 151, "end": 248},
                 {"name": "Pkinase", "start": 270, "end": 520}]},
    {"name": "Abl", "length": 1149,
     "domains": [{"name": "SH3", "start": 66, "end": 118},
                 {"name": "SH2", "start": 126, "end": 218},
                 {"name": "Pkinase", "start": 242, "end": 493}]},
    {"name": "EGFR", "length": 1210,
     "domains": [{"name": "Recep_L", "start": 1, "end": 165},
                 {"name": "Furin-like", "start": 166, "end": 309},
                 {"name": "Recep_L", "start": 310, "end": 480},
                 {"name": "GF_recep_IV", "start": 481, "end": 620},
                 {"name": "Pkinase", "start": 712, "end": 968}]},
]

compare_architectures(kinase_family)

---

## 8. Biological Applications

### 8.1 Finding Transcription Factor Binding Sites

Given a set of known binding sites, build a PWM and scan a promoter region.

In [ ]:
# CRP (cAMP receptor protein) binding sites from E. coli
crp_sites = [
    "AAATGTGATCTAGATCACATTT",
    "GAATGTGATCTATATCACATTT",
    "CAATGTGATCGAGATCACATAA",
    "TAATGTGATCTTAATCACATAT",
    "AAATGTGATCTGGATCACATTT",
    "GAATGTGAAATAGATCACATTT",
    "AAATGTGATTTAGATCACATTT",
    "CAATGTGATCTCAATCACATCA",
]

# Build PWM for CRP
crp_pfm = build_pfm(crp_sites)
crp_ppm = pfm_to_ppm(crp_pfm, pseudocount=0.5)
crp_pwm = ppm_to_pwm(crp_ppm)

print(f"CRP motif: {len(crp_sites[0])} bp, built from {len(crp_sites)} sites")
print(f"Consensus: {''.join(BASES[i] for i in np.argmax(crp_ppm, axis=0))}")

# Show logo
plot_sequence_logo(crp_ppm, "CRP Binding Site Logo")

In [ ]:
# Scan a promoter for CRP sites
lac_promoter = (
    "GACACCATCGAATGGCGCAAAACCTTTCGCGGTATGGCATGATAGCGCCC"
    "GGAAGAGAGTCAATTCAGGGTGGTGAATGTGAAACCAGTAACGTTATAC"
    "GATGTCGCAGAGTATGCCGGTGTCTCTTATCAGACCGTTTCCCGCGTGGT"
    "GAACCAGGCCAGCCACGTTTCTGCGAAAACGCGGGAAAAAGTGGAAGCGG"
)

max_crp = np.sum(np.max(crp_pwm, axis=0))
crp_hits = scan_both_strands(crp_pwm, lac_promoter,
                              threshold=0.50 * max_crp)

print(f"Scanning {len(lac_promoter)} bp promoter for CRP binding sites")
print(f"Threshold: 50% of max score ({0.50 * max_crp:.1f})\n")

print(f"{'Pos':>4} {'Strand':>6} {'Match':<24} {'Score':>7} {'% max':>6}")
print("-" * 52)
for pos, subseq, score, strand in crp_hits[:5]:
    print(f"{pos:4d} {strand:>6} {subseq:<24} {score:7.1f} {100 * score / max_crp:5.1f}%")

### 8.2 Identifying Protein Domains in a Query Sequence

In practice you would run `hmmscan` or InterProScan. Here we simulate the workflow
by building simple profile-based scoring for demonstration.

In [ ]:
def build_simple_profile(aligned_seqs):
    """
    Build a position-specific scoring profile from aligned protein sequences.
    Returns a list of (consensus_residue, conservation_fraction) per position.
    """
    length = len(aligned_seqs[0])
    profile = []

    for pos in range(length):
        residues = [s[pos] for s in aligned_seqs if s[pos] != "-"]
        if not residues:
            profile.append(("-", 0.0))
            continue
        counts = Counter(residues)
        top_aa, top_count = counts.most_common(1)[0]
        conservation = top_count / len(residues)
        profile.append((top_aa, conservation))

    return profile


def score_against_profile(sequence, profile, gap_penalty=-2):
    """
    Score a sequence against a simple profile.
    Match at conserved position: +2 * conservation
    Mismatch at conserved position: -1
    Variable position: +0.5
    """
    score = 0.0
    for i, (cons_aa, cons) in enumerate(profile):
        if i >= len(sequence):
            break
        if cons > 0.5:
            if sequence[i] == cons_aa:
                score += 2 * cons
            else:
                score -= 1
        else:
            score += 0.5
    return score


# Example: build profile for SH2-like domain
sh2_alignment = [
    "FLVRESETTK",
    "YLVRESETAL",
    "FLVRESQSTK",
    "WLVKESETTK",
    "FLVRESETTK",
]

sh2_profile = build_simple_profile(sh2_alignment)
print("SH2-like domain profile:")
for i, (aa, cons) in enumerate(sh2_profile):
    bar = '#' * int(cons * 20)
    print(f"  Position {i + 1}: {aa} ({cons:.0%}) {bar}")

# Score test sequences
test_seqs = ["FLVRESETTK", "FLAAESETTK", "GGGGGGGGGG"]
for seq in test_seqs:
    s = score_against_profile(seq, sh2_profile)
    print(f"  {seq}: score = {s:.1f}")

---

## 9. Motif Databases and File Formats

### JASPAR Format

The most widely used TF motif database. Matrices are stored as count matrices:

```
>MA0004.1 Arnt
A [  4 19  0  0  0  0 ]
C [  4  0  0  0  0  0 ]
G [  3  0 23  0 23  0 ]
T [ 12  4  0 23  0 23 ]
```

### MEME Format

Used by the MEME Suite motif analysis tools:

```
MOTIF MA0004.1 Arnt
letter-probability matrix: alength= 4 w= 6
 0.174  0.174  0.130  0.522
 0.826  0.000  0.000  0.174
 ...
```

In [ ]:
def parse_jaspar(text):
    """
    Parse a JASPAR-format count matrix.
    Returns (name, pfm_array).
    """
    lines = text.strip().split("\n")
    name = lines[0].lstrip(">").strip() if lines[0].startswith(">") else "Unknown"

    counts = {}
    for line in lines[1:]:
        m = re.match(r'([ACGT])\s*\[([\d\s.]+)\]', line)
        if m:
            base = m.group(1)
            values = [float(x) for x in m.group(2).split()]
            counts[base] = values

    if not counts:
        raise ValueError("Could not parse JASPAR matrix")

    pfm = np.array([counts[b] for b in BASES])
    return name, pfm


jaspar_text = """>MA0004.1 Arnt
A [  4 19  0  0  0  0 ]
C [  4  0  0  0  0  0 ]
G [  3  0 23  0 23  0 ]
T [ 12  4  0 23  0 23 ]"""

name, arnt_pfm = parse_jaspar(jaspar_text)
arnt_ppm = pfm_to_ppm(arnt_pfm, pseudocount=0.5)

print(f"Parsed: {name}")
print(f"Length: {arnt_pfm.shape[1]} positions")
print(f"Consensus: {''.join(BASES[i] for i in np.argmax(arnt_ppm, axis=0))}")

plot_sequence_logo(arnt_ppm, f"Sequence Logo: {name}")

---

## Exercises

### Exercise 1: Build a PWM from Aligned Sequences (Difficulty: *)

Given these E-box binding sites, build a PFM, convert to PPM and PWM, and print
the consensus sequence.

```python
ebox_sites = [
    "CACGTG",
    "CACGTG",
    "CATGTG",
    "CACGTG",
    "CACGCG",
    "CACGTG",
    "CATGTG",
    "CACGTG",
]
```

In [ ]:
# Exercise 1 -- your code here
ebox_sites = [
    "CACGTG",
    "CACGTG",
    "CATGTG",
    "CACGTG",
    "CACGCG",
    "CACGTG",
    "CATGTG",
    "CACGTG",
]


In [ ]:
# --- Solution ---
ebox_pfm = build_pfm(ebox_sites)
ebox_ppm = pfm_to_ppm(ebox_pfm, pseudocount=0.1)
ebox_pwm = ppm_to_pwm(ebox_ppm)

print("PFM (counts):")
for i, base in enumerate(BASES):
    print(f"  {base}: {list(ebox_pfm[i].astype(int))}")

consensus = "".join(BASES[i] for i in np.argmax(ebox_ppm, axis=0))
print(f"\nConsensus: {consensus}")

ic = information_content(ebox_ppm)
print(f"Total IC: {ic.sum():.2f} bits")

display_ppm_heatmap(ebox_ppm, "E-box PPM")

### Exercise 2: Scan a Sequence for Motif Hits (Difficulty: *)

Using the E-box PWM from Exercise 1, scan this sequence and report all hits
above 70% of the maximum score:

```python
seq = "ATCGCACGTGAAATTTCATGTGCCCGGGCACGTGATCACGCGAATT"
```

In [ ]:
# Exercise 2 -- your code here
seq = "ATCGCACGTGAAATTTCATGTGCCCGGGCACGTGATCACGCGAATT"


In [ ]:
# --- Solution ---
max_ebox = np.sum(np.max(ebox_pwm, axis=0))
threshold = 0.70 * max_ebox

hits_ex2 = scan_sequence(ebox_pwm, seq, threshold=threshold)

print(f"Sequence: {seq}")
print(f"Threshold: {threshold:.2f} (70% of max {max_ebox:.2f})\n")
print(f"{'Pos':>4} {'Match':<8} {'Score':>7} {'% max':>6}")
print("-" * 30)
for pos, subseq, score in hits_ex2:
    print(f"{pos:4d} {subseq:<8} {score:7.2f} {100 * score / max_ebox:5.1f}%")

### Exercise 3: Calculate and Plot Information Content (Difficulty: **)

Calculate the per-position information content of the CRP motif (defined in Section 8.1)
and create a bar chart showing IC at each position. Which positions are most conserved?

In [ ]:
# Exercise 3 -- your code here


In [ ]:
# --- Solution ---
crp_ic = information_content(crp_ppm)

fig, ax = plt.subplots(figsize=(10, 3))
positions = np.arange(1, len(crp_ic) + 1)
colors = ["#2c7bb6" if v > 1.0 else "#abd9e9" if v > 0.5 else "#fdae61" for v in crp_ic]
ax.bar(positions, crp_ic, color=colors, edgecolor="none")
ax.set_xlabel("Position")
ax.set_ylabel("Information content (bits)")
ax.set_title("CRP Motif -- Information Content per Position")
ax.set_ylim(0, 2.1)
ax.axhline(1.0, color="gray", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

# Most conserved positions
top_5 = np.argsort(crp_ic)[::-1][:5]
print("Top 5 most conserved positions:")
for idx in top_5:
    best_base = BASES[np.argmax(crp_ppm[:, idx])]
    print(f"  Position {idx + 1}: IC = {crp_ic[idx]:.2f} bits (dominant base: {best_base})")

### Exercise 4: PROSITE Pattern Scanning (Difficulty: **)

Scan the following protein sequence for:
1. N-glycosylation sites: `N-{P}-[ST]-{P}`
2. Casein kinase II phosphorylation sites: `[ST]-x(2)-[DE]`
3. Myristoylation sites: `G-{EDRKHPFYW}-x(2)-[STAGCN]-{P}`

```python
protein = "MGSSHHHHHHSSGLVPRGSHMRGNATFVFNLTPEEDKDLYIDSTTNLTQNFAKWEDN"
```

In [ ]:
# Exercise 4 -- your code here
protein = "MGSSHHHHHHSSGLVPRGSHMRGNATFVFNLTPEEDKDLYIDSTTNLTQNFAKWEDN"


In [ ]:
# --- Solution ---
prosite_patterns = {
    "N-glycosylation": "N-{P}-[ST]-{P}",
    "CK2 phosphorylation": "[ST]-x(2)-[DE]",
    "Myristoylation": "G-{EDRKHPFYW}-x(2)-[STAGCN]-{P}",
}

print(f"Sequence ({len(protein)} aa): {protein}\n")
for name, pat in prosite_patterns.items():
    scan_prosite(protein, pat, name)
    print()

### Exercise 5: Visualize Domain Architecture (Difficulty: **)

Create domain architecture diagrams for these two proteins and then use
`compare_architectures` to compare them:

1. **JAK2** (1132 aa): FERM domain (37-380), SH2 domain (401-482), Pseudokinase (510-827), Kinase (849-1124)
2. **JAK3** (1124 aa): FERM domain (35-378), SH2 domain (399-480), Pseudokinase (505-820), Kinase (841-1116)

In [ ]:
# Exercise 5 -- your code here


In [ ]:
# --- Solution ---
jak2_domains = [
    {"name": "FERM", "start": 37, "end": 380},
    {"name": "SH2", "start": 401, "end": 482},
    {"name": "Pseudokinase", "start": 510, "end": 827},
    {"name": "Kinase", "start": 849, "end": 1124},
]

jak3_domains = [
    {"name": "FERM", "start": 35, "end": 378},
    {"name": "SH2", "start": 399, "end": 480},
    {"name": "Pseudokinase", "start": 505, "end": 820},
    {"name": "Kinase", "start": 841, "end": 1116},
]

visualize_domain_architecture("JAK2", 1132, jak2_domains)
visualize_domain_architecture("JAK3", 1124, jak3_domains)

compare_architectures([
    {"name": "JAK2", "length": 1132, "domains": jak2_domains},
    {"name": "JAK3", "length": 1124, "domains": jak3_domains},
])

### Exercise 6: Parse JASPAR and Scan (Difficulty: ***)

Parse this JASPAR matrix for the GATA1 transcription factor, build a PWM, and
scan the given sequence for hits above 60% of the max score on both strands.

```python
gata1_jaspar = """>MA0035.1 GATA1
A [  1  0 26  0  0 14 ]
C [  0  0  0  0  0  3 ]
G [  0 26  0 26  0  5 ]
T [ 25  0  0  0 26  4 ]"""

promoter = "CCAGATAAGATCTTTTATTTATCCCCC"
```

In [ ]:
# Exercise 6 -- your code here
gata1_jaspar = """>MA0035.1 GATA1
A [  1  0 26  0  0 14 ]
C [  0  0  0  0  0  3 ]
G [  0 26  0 26  0  5 ]
T [ 25  0  0  0 26  4 ]"""

promoter = "CCAGATAAGATCTTTTATTTATCCCCC"


In [ ]:
# --- Solution ---
gata_name, gata_pfm = parse_jaspar(gata1_jaspar)
gata_ppm = pfm_to_ppm(gata_pfm, pseudocount=0.5)
gata_pwm = ppm_to_pwm(gata_ppm)

print(f"Motif: {gata_name}")
print(f"Consensus: {''.join(BASES[i] for i in np.argmax(gata_ppm, axis=0))}")

plot_sequence_logo(gata_ppm, f"Sequence Logo: {gata_name}")

max_gata = np.sum(np.max(gata_pwm, axis=0))
gata_hits = scan_both_strands(gata_pwm, promoter, threshold=0.60 * max_gata)

print(f"\nScanning: {promoter}")
print(f"Threshold: 60% of max ({0.60 * max_gata:.2f})\n")
print(f"{'Pos':>4} {'Strand':>6} {'Match':<8} {'Score':>7} {'% max':>6}")
print("-" * 36)
for pos, subseq, score, strand in gata_hits:
    print(f"{pos:4d} {strand:>6} {subseq:<8} {score:7.2f} {100 * score / max_gata:5.1f}%")

---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| Sequence motifs | Short conserved patterns with biological function |
| PWM | Quantitative motif model: PFM -> PPM (+ pseudocount) -> log-odds PWM |
| Information content | IC = 2 - H(pos); max 2 bits for DNA = perfectly conserved |
| Sequence logo | Visual encoding: stack height = IC, letter height = frequency * IC |
| PROSITE | Regex-like patterns for protein motifs |
| Protein domains | Independent structural/functional units; detected by profile HMMs |
| Pfam / InterPro | Major domain databases; use hmmscan or InterProScan |
| Domain architecture | Arrangement of domains defines protein function |

---

## Resources

- [JASPAR database](https://jaspar.elixir.no/) -- transcription factor binding profiles
- [MEME Suite](https://meme-suite.org/) -- motif analysis toolkit
- [WebLogo](https://weblogo.berkeley.edu/) -- publication-quality sequence logos
- [PROSITE](https://prosite.expasy.org/) -- protein patterns and profiles
- [Pfam (via InterPro)](https://www.ebi.ac.uk/interpro/entry/pfam/) -- protein families
- [InterPro](https://www.ebi.ac.uk/interpro/) -- integrated domain/family annotations
- [HMMER](http://hmmer.org/) -- profile HMM search software
- [InterProScan](https://www.ebi.ac.uk/interpro/search/sequence/) -- online domain search
- Stormo GD (2000). DNA binding sites: representation and discovery. *Bioinformatics* 16:16-23

---

[< Previous: Sequence Motifs + Domains: Overview](01_sequence_motifs_and_domains.ipynb) | [Tier 2: Core Bioinformatics Overview](../README.md) | [Next: Gene Ontology >](../11_Gene_Ontology_and_Pathways/01_gene_ontology.ipynb)

---